In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from scripts.plotting import *

def make_swiss_roll(n_samples=100, *, noise=0.0, random_state=None, hole=False):
    """Generate a swiss roll dataset.

    Read more in the :ref:`User Guide <sample_generators>`.

    Parameters
    ----------
    n_samples : int, default=100
        The number of sample points on the Swiss Roll.

    noise : float, default=0.0
        The standard deviation of the gaussian noise.

    random_state : int, RandomState instance or None, default=None
        Determines random number generation for dataset creation. Pass an int
        for reproducible output across multiple function calls.
        See :term:`Glossary <random_state>`.

    hole : bool, default=False
        If True generates the swiss roll with hole dataset.

    Returns
    -------
    X : ndarray of shape (n_samples, 3)
        The points.

    t : ndarray of shape (n_samples,)
        The univariate position of the sample according to the main dimension
        of the points in the manifold.

    Notes
    -----
    The algorithm is from Marsland [1].

    References
    ----------
    .. [1] S. Marsland, "Machine Learning: An Algorithmic Perspective", 2nd edition,
           Chapter 6, 2014.
           https://homepages.ecs.vuw.ac.nz/~marslast/Code/Ch6/lle.py

    Examples
    --------
    >>> from sklearn.datasets import make_swiss_roll
    >>> X, t = make_swiss_roll(noise=0.05, random_state=0)
    >>> X.shape
    (100, 3)
    >>> t.shape
    (100,)
    """
    rng = np.random.default_rng(random_state)

    if not hole:
        t = 1.5 * np.pi * (1 + 2 * rng.uniform(size=n_samples))
        y = 21 * rng.uniform(size=n_samples)
    else:
        corners = np.array(
            [[np.pi * (1.5 + i), j * 7] for i in range(3) for j in range(3)]
        )
        corners = np.delete(corners, 4, axis=0)
        corner_index = rng.choice(8, n_samples)
        parameters = rng.uniform(size=(2, n_samples)) * np.array([[np.pi], [7]])
        t, y = corners[corner_index].T + parameters

    x = t * np.cos(t)
    z = t * np.sin(t)

    X = np.vstack((x, y, z))
    X += noise * rng.standard_normal(size=(3, n_samples))
    X = X.T
    t = np.squeeze(t)

    return X, t


def compute_swiss_roll_derivative(t):
    """
    Compute the derivative (tangent vectors) of the Swiss Roll.

    Parameters
    ----------
    t : ndarray of shape (n_samples,)
        The univariate position parameter of the Swiss Roll.

    Returns
    -------
    V : ndarray of shape (n_samples, 3)
        The tangent vectors at each point on the Swiss Roll.
    """
    dx0_dt = np.cos(t) - t * np.sin(t)
    dx1_dt = np.sin(t) + t * np.cos(t)
    dx2_dt = np.zeros_like(t)  # y is independent

    dx0_dy = np.zeros_like(t)    # Constant component for y (2.0 * uniform), derivative is zero
    dx1_dy = 2 * np.ones_like(t)     # Constant component for y (2.0 * uniform), derivative is 1
    dx2_dy = np.zeros_like(t)    # Constant component for y (2.0 * uniform), derivative is zero
    
    dX_dt = np.stack([dx0_dt, dx1_dt, dx2_dt], axis=1)
    dX_dy = np.stack([dx0_dy, dx1_dy, dx2_dy], axis=1)
    
    return dX_dt, dX_dy


def compute_general_derivative(t, dt_ds):
    """
    Compute the derivative of the S curve with respect to a general parameter s,
    where t is a function of s.

    Parameters
    ----------
    t : ndarray
        The values of t for the S curve.

    dt_ds : ndarray
        The derivative of t with respect to s (dt/ds).

    Returns
    -------
    dX_ds : ndarray of shape (n_samples, 3)
        The derivative of the S curve with respect to s for each coordinate.
    """
    # Compute derivatives for each component with respect to t
    dx0_dt = np.cos(t)          # Derivative of sin(t) with respect to t
    dx1_dt = np.zeros_like(t)   # y-coordinate derivative is zero (constant in this case)
    dx2_dt = -np.sin(t) * np.sign(t)  # Derivative of z-component with respect to t

    # Stack derivatives to form dX/dt
    dX_dt = np.stack([dx0_dt, dx1_dt, dx2_dt], axis=1)

    # Apply chain rule to compute dX/ds
    dX_ds = dX_dt * dt_ds[:, np.newaxis]  # Multiply by dt/ds for each coordinate
    dY_ds = None
    return dX_ds, dY_ds

In [ ]:
X, t = make_swiss_roll(1000, random_state=42)
plot_3d(X, t, "")

In [ ]:
def normalize_rows(dX_dt):
    # Compute the L2 norm for each row (with keepdims for proper broadcasting)
    row_norms = np.linalg.norm(dX_dt, axis=1, keepdims=True)
    
    # Avoid division by zero: if a row's norm is zero, set it to 1 (so the row stays unchanged)
    row_norms[row_norms == 0] = 1
    
    # Divide each element in the row by the corresponding row norm
    return dX_dt / row_norms

In [ ]:
# Compute derivatives with respect to s
dX_dt, dX_dy = compute_swiss_roll_derivative(t)
dX_ds = normalize_rows(dX_dt)
plot_3d_with_quiver(X, dX_ds, t, arrow_size=1.5, normalize=False, title="3D swiss roll with uniform vector field")

In [ ]:
# Define file names
position_matrix_file = "./data/swiss_roll/uniform/swiss_roll_gt_position_matrix.csv"
velocity_matrix_file = "./data/swiss_roll/uniform/swiss_roll_gt_velocity_matrix.csv"
latent_time_vector_file = "./data/swiss_roll/uniform/swiss_roll_gt_latent_time_vector.csv"

# Convert data to DataFrames for saving
X_df = pd.DataFrame(X, columns=["X", "Y", "Z"])
dX_ds_df = pd.DataFrame(dX_dt, columns=["dX/ds_X", "dX/ds_Y", "dX/ds_Z"])
t_df = pd.DataFrame(t, columns=["t"])

# Write to CSV files
X_df.to_csv(position_matrix_file, index=False)
dX_ds_df.to_csv(velocity_matrix_file, index=False)
t_df.to_csv(latent_time_vector_file, index=False)

(position_matrix_file, velocity_matrix_file, latent_time_vector_file)

In [ ]:
# Set seed for reproducibility
np.random.seed(42)

# Define noise level
position_noise_std = 1.5
velocity_noise_std = 1

# Add Gaussian noise to the position and velocity matrices
noisy_points = X + np.random.normal(scale=position_noise_std, size=X.shape)
noisy_dX_ds = dX_ds + np.random.normal(scale=velocity_noise_std, size=dX_ds.shape)

# Define file names for noisy data
noisy_position_matrix_file = position_matrix_file.replace("gt", "noisy")
noisy_velocity_matrix_file = velocity_matrix_file.replace("gt", "noisy")

# Convert noisy data to DataFrames for saving
noisy_X_df = pd.DataFrame(noisy_points, columns=["X", "Y", "Z"])
noisy_dX_ds_df = pd.DataFrame(noisy_dX_ds, columns=["dX/ds_X", "dX/ds_Y", "dX/ds_Z"])

# Write noisy data to CSV files
noisy_X_df.to_csv(noisy_position_matrix_file, index=False)
noisy_dX_ds_df.to_csv(noisy_velocity_matrix_file, index=False)

(noisy_position_matrix_file, noisy_velocity_matrix_file)

In [ ]:
plot_3d_with_quiver(noisy_points, noisy_dX_ds, t, arrow_size=2.5, title="3D S-Curve with vector field (noisy)")

In [ ]:
# Define the dimensions for projection
high_dim = 100
np.random.seed(42)  # Ensure reproducibility

# Generate a random projection matrix
projection_matrix = np.random.normal(size=(3, high_dim))

# Project the clean data to 100 dimensions
high_dim_points = X @ projection_matrix  # Shape: (n_samples, high_dim)
high_dim_velocity = dX_ds @ projection_matrix  # Shape: (n_samples, high_dim)

# Add Gaussian noise to the high-dimensional data
high_dim_points_noisy = high_dim_points + np.random.normal(scale=10, size=high_dim_points.shape)
high_dim_velocity_noisy = high_dim_velocity + np.random.normal(scale=10, size=high_dim_velocity.shape)

# Define file names for high-dimensional noisy data
noisy_hd_position_matrix_file = position_matrix_file.replace("gt", "noisy_hd")
noisy_hd_velocity_matrix_file = velocity_matrix_file.replace("gt", "noisy_hd")

# Save high-dimensional noisy data
pd.DataFrame(high_dim_points_noisy).to_csv(noisy_hd_position_matrix_file, index=False, header=[f"dim_{i}" for i in range(high_dim)])
pd.DataFrame(high_dim_velocity_noisy).to_csv(noisy_hd_velocity_matrix_file, index=False, header=[f"dim_{i}" for i in range(high_dim)])

(noisy_hd_position_matrix_file, noisy_hd_velocity_matrix_file)


In [ ]:
from sklearn.decomposition import PCA

# Perform PCA on the high-dimensional noisy position matrix
pca = PCA(n_components=3)
principal_components = pca.fit_transform(high_dim_points_noisy)  # Shape: (n_samples, 3)

# Project the high-dimensional velocity matrix using the same PCA
velocity_components = high_dim_velocity_noisy @ pca.components_.T  # Shape: (n_samples, 3)

# Visualize using the provided function
plot_3d_with_quiver(
    principal_components,
    velocity_components,
    t,
    arrow_size=10,
    title="3D S-Curve with vector field (HD noisy)",
)